# Article 3: RAG Evaluation Framework - Analysis & Visualization

This notebook analyzes and visualizes evaluation metrics from Article 3:
- RAGAS metrics (groundedness, answer_correctness, faithfulness)
- DeepEval metrics (contextual_precision)
- LLM-as-judge correlation with golden set
- Drift detection simulation

**Outputs**: Charts exported to `results/charts/article_03/`

In [ ]:
import json
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Create output directory
output_dir = Path("../results/charts/article_03")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {output_dir}")

## 1. Correlation Analysis: Groundedness vs Simulated User Rating

Test hypothesis: RAGAS groundedness score correlates with user satisfaction

In [ ]:
# Generate simulated data (replace with actual evaluation results)
np.random.seed(42)

# Simulate groundedness scores (0-1) and user ratings (1-5)
# Positive correlation: higher groundedness = higher rating
n_samples = 100
groundedness = np.random.beta(5, 2, n_samples)  # Skewed toward high scores
# User rating correlates with groundedness + some noise
user_rating = 1 + groundedness * 3.5 + np.random.normal(0, 0.5, n_samples)
user_rating = np.clip(user_rating, 1, 5)  # Clip to valid range

# Calculate Pearson correlation
corr_coef, p_value = stats.pearsonr(groundedness, user_rating)

# Create scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(groundedness, user_rating, alpha=0.6, s=50)

# Add regression line
z = np.polyfit(groundedness, user_rating, 1)
p = np.poly1d(z)
plt.plot(groundedness, p(groundedness), "r--", alpha=0.8, linewidth=2)

plt.xlabel("RAGAS Groundedness Score", fontsize=12)
plt.ylabel("Simulated User Rating (1-5)", fontsize=12)
plt.title(f"Correlation: Groundedness vs User Rating\n(r={corr_coef:.3f}, p={p_value:.4f})", fontsize=14)
plt.grid(True, alpha=0.3)

# Add text annotation
plt.text(0.05, 4.5, f"Pearson r = {corr_coef:.3f}\np-value = {p_value:.4f}\n" +
         ("Significant" if p_value < 0.05 else "Not significant"), 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5), fontsize=10)

plt.tight_layout()
plt.savefig(output_dir / "01_groundedness_vs_rating.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"Correlation coefficient: {corr_coef:.3f}")
print(f"P-value: {p_value:.4f}")
print(f"Interpretation: {'Significant' if p_value < 0.05 else 'Not significant'} correlation")

## 2. Metric Stability: Variance Across Multiple Runs

Test evaluation metric consistency across 3 runs

In [ ]:
# Simulate 3 runs of evaluation metrics
metrics = ['Groundedness', 'Answer\nCorrectness', 'Faithfulness', 'Context\nPrecision']
n_runs = 3

# Generate simulated data with realistic variance
np.random.seed(43)
data = {
    'Groundedness': np.random.normal(0.85, 0.03, n_runs),
    'Answer\nCorrectness': np.random.normal(0.78, 0.04, n_runs),
    'Faithfulness': np.random.normal(0.82, 0.035, n_runs),
    'Context\nPrecision': np.random.normal(0.75, 0.045, n_runs),
}

# Calculate mean and std
means = [np.mean(data[m]) for m in metrics]
stds = [np.std(data[m], ddof=1) for m in metrics]

# Create bar chart with error bars
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(metrics))
bars = ax.bar(x, means, yerr=stds, capsize=10, alpha=0.7, 
               color=['#2E86AB', '#A23B72', '#F18F01', '#06A77D'])

ax.set_xlabel("Evaluation Metric", fontsize=12)
ax.set_ylabel("Score (0-1)", fontsize=12)
ax.set_title("Metric Stability Across 3 Runs (mean ± std)", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (bar, mean, std) in enumerate(zip(bars, means, stds)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + std + 0.02,
            f'{mean:.3f}±{std:.3f}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(output_dir / "02_metric_stability.png", dpi=300, bbox_inches='tight')
plt.show()

print("\nMetric Stability Report:")
for metric, mean, std in zip(metrics, means, stds):
    cv = (std / mean) * 100  # Coefficient of variation
    print(f"{metric:25s}: {mean:.3f} ± {std:.3f} (CV: {cv:.1f}%)")

## 3. Drift Detection Simulation

Visualize KL divergence over time with drift threshold

In [ ]:
# Simulate drift detection timeline
np.random.seed(44)

# Normal period: low KL divergence
normal_period = np.random.gamma(2, 0.02, 500)  # Low values

# Drift period: increasing KL divergence
drift_start = 500
drift_period = np.random.gamma(3, 0.05, 200) + 0.05  # Higher values

# Combine
kl_divergence = np.concatenate([normal_period, drift_period])
timestamps = np.arange(len(kl_divergence))

# Create plot
fig, ax = plt.subplots(figsize=(12, 6))

# Plot KL divergence
ax.plot(timestamps, kl_divergence, linewidth=1, alpha=0.7, label='KL Divergence')

# Plot threshold
threshold = 0.15
ax.axhline(y=threshold, color='r', linestyle='--', linewidth=2, label=f'Alert Threshold ({threshold})')

# Highlight drift region
ax.axvspan(drift_start, len(kl_divergence), alpha=0.2, color='red', label='Drift Detected')

# Mark first alert
first_alert_idx = np.where(kl_divergence > threshold)[0]
if len(first_alert_idx) > 0:
    first_alert = first_alert_idx[0]
    ax.plot(first_alert, kl_divergence[first_alert], 'r*', markersize=15, 
            label=f'First Alert (sample {first_alert})')

ax.set_xlabel("Query Number", fontsize=12)
ax.set_ylabel("KL Divergence", fontsize=12)
ax.set_title("Embedding Drift Detection Over Time", fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / "03_drift_detection_timeline.png", dpi=300, bbox_inches='tight')
plt.show()

# Statistics
alerts = np.sum(kl_divergence > threshold)
alert_rate = alerts / len(kl_divergence) * 100
print(f"\nDrift Statistics:")
print(f"Total queries: {len(kl_divergence)}")
print(f"Alerts triggered: {alerts} ({alert_rate:.1f}%)")
print(f"Mean KL (normal): {np.mean(kl_divergence[:drift_start]):.4f}")
print(f"Mean KL (drift): {np.mean(kl_divergence[drift_start:]):.4f}")

## 4. LLM-as-Judge vs Golden Set Agreement

Confusion matrix showing agreement between automated judge and human-verified golden set

In [ ]:
# Simulate LLM judge vs golden set comparison
# Classes: Good (1), Acceptable (0), Poor (-1)
np.random.seed(45)

n_samples = 50  # From golden set

# Golden set labels (ground truth)
golden_labels = np.random.choice([1, 0, -1], n_samples, p=[0.4, 0.35, 0.25])

# LLM judge predictions (mostly agree, some errors)
llm_predictions = golden_labels.copy()
# Introduce 15% disagreement
disagreement_idx = np.random.choice(n_samples, int(n_samples * 0.15), replace=False)
for idx in disagreement_idx:
    llm_predictions[idx] = np.random.choice([l for l in [1, 0, -1] if l != golden_labels[idx]])

# Create confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(golden_labels, llm_predictions, labels=[1, 0, -1])

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Good', 'Acceptable', 'Poor'],
            yticklabels=['Good', 'Acceptable', 'Poor'],
            ax=ax, cbar_kws={'label': 'Count'})

ax.set_xlabel("LLM Judge Prediction", fontsize=12)
ax.set_ylabel("Golden Set (Ground Truth)", fontsize=12)
ax.set_title("LLM-as-Judge Agreement with Golden Set", fontsize=14)

plt.tight_layout()
plt.savefig(output_dir / "04_llm_judge_confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.show()

# Calculate agreement metrics
accuracy = np.sum(golden_labels == llm_predictions) / len(golden_labels)
print(f"\nAgreement Metrics:")
print(f"Accuracy: {accuracy:.1%}")
print(f"\nClassification Report:")
print(classification_report(golden_labels, llm_predictions, 
                            target_names=['Good', 'Acceptable', 'Poor']))

## 5. Summary & Key Findings

Aggregated insights from evaluation framework

In [ ]:
print("="*70)
print("Article 3: RAG Evaluation Framework - Key Findings")
print("="*70)

print("\n1. Correlation Analysis:")
print(f"   - Groundedness vs User Rating: r={corr_coef:.3f}, p={p_value:.4f}")
print(f"   - Interpretation: {'Strong' if abs(corr_coef) > 0.7 else 'Moderate' if abs(corr_coef) > 0.4 else 'Weak'} positive correlation")
print(f"   - Statistical significance: {'Yes' if p_value < 0.05 else 'No'} (p<0.05)")

print("\n2. Metric Stability:")
for metric, mean, std in zip(metrics, means, stds):
    cv = (std / mean) * 100
    stability = "Excellent" if cv < 3 else "Good" if cv < 5 else "Fair"
    print(f"   - {metric:25s}: {mean:.3f}±{std:.3f} ({stability}, CV={cv:.1f}%)")

print("\n3. Drift Detection:")
print(f"   - Alert rate: {alert_rate:.1f}%")
print(f"   - Mean KL divergence (baseline): {np.mean(kl_divergence[:drift_start]):.4f}")
print(f"   - Mean KL divergence (drift): {np.mean(kl_divergence[drift_start:]):.4f}")
print(f"   - Detection threshold: {threshold}")

print("\n4. LLM-as-Judge Validation:")
print(f"   - Agreement with golden set: {accuracy:.1%}")
print(f"   - Suitable for: {'Production use' if accuracy > 0.8 else 'Assisted evaluation only'}")

print("\n5. Recommendations:")
print("   - Use RAGAS groundedness as primary metric (correlates with satisfaction)")
print("   - Run evaluations 3x and report mean±std for reliability")
print("   - Monitor KL divergence weekly for production drift detection")
print(f"   - {'Trust' if accuracy > 0.85 else 'Supplement'} LLM-as-judge with spot-checks")

print("\n" + "="*70)
print(f"Charts exported to: {output_dir.resolve()}")
print("="*70)

## 6. Export Summary Statistics

Save analysis results as JSON for programmatic access

In [ ]:
summary_data = {
    "correlation_analysis": {
        "groundedness_vs_rating": {
            "pearson_r": float(corr_coef),
            "p_value": float(p_value),
            "significant": p_value < 0.05,
            "interpretation": "Strong" if abs(corr_coef) > 0.7 else "Moderate" if abs(corr_coef) > 0.4 else "Weak"
        }
    },
    "metric_stability": {
        metric: {
            "mean": float(mean),
            "std": float(std),
            "cv_percent": float((std / mean) * 100)
        }
        for metric, mean, std in zip(metrics, means, stds)
    },
    "drift_detection": {
        "total_queries": int(len(kl_divergence)),
        "alerts_triggered": int(alerts),
        "alert_rate_percent": float(alert_rate),
        "threshold": float(threshold),
        "mean_kl_baseline": float(np.mean(kl_divergence[:drift_start])),
        "mean_kl_drift": float(np.mean(kl_divergence[drift_start:]))
    },
    "llm_judge_validation": {
        "accuracy": float(accuracy),
        "total_samples": int(n_samples),
        "production_ready": accuracy > 0.8
    }
}

summary_path = Path("../results/data/article_03_summary.json")
summary_path.parent.mkdir(parents=True, exist_ok=True)
with open(summary_path, 'w') as f:
    json.dump(summary_data, f, indent=2)

print(f"Summary data exported to: {summary_path.resolve()}")